# Data Wrangling Capstone: Gather → Assess → Clean
### Credit Card Risk Analysis Track

Every notebook so far taught you individual pandas tools. This one teaches the **workflow** that ties them together — the actual process a data analyst follows on a messy, real dataset, start to finish:

1. **Gather** — get the data into pandas, from however many sources it comes in.
2. **Assess** — look at the data (visually and programmatically) and **write down every problem you find**, before fixing anything. Problems come in two flavors:
   - **Quality issues**: content problems — missing values, duplicates, wrong dtypes, inconsistent categories, invalid/implausible values.
   - **Tidiness issues**: structural problems — data split across tables that should be one, or a single column secretly holding two different variables.
3. **Clean** — for each issue you documented, follow **Define → Code → Test**: define in words what you're about to do and why, write the code, then test that it actually worked before moving to the next issue.

**Datasets:** `loan_applications_wrangle.csv` (a fresh, deliberately messy export — similar to Phase 3's raw file, but with two new problems added) and `credit_bureau.csv` (the same external bureau file from Phase 6). Keep both in the same folder as this notebook.

**Format note:** this notebook is different from earlier ones. The Assess section asks you to write your **own** issue list in a markdown cell before seeing mine — that act of writing it down is the actual skill being practiced, not just running code. The Clean section still has `YOUR CODE HERE`/`Solution` pairs, but each one follows Define → Code → Test instead of a plain question.

All solution code was run end-to-end against the actual datasets before this notebook was assembled.

## Setup

In [ ]:
import pandas as pd
import numpy as np
print(pd.__version__)

## Part 1: Gather

Gathering isn't always "read one CSV." Real projects pull from multiple sources that have to be reconciled later — that reconciliation is itself one of the issues you'll document in Assess.

**G1.** Load `loan_applications_wrangle.csv` into `loans` (parse `application_date` as a date) and `credit_bureau.csv` into `bureau`. Print both shapes.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
loans = pd.read_csv("loan_applications_wrangle.csv", parse_dates=["application_date"])
bureau = pd.read_csv("credit_bureau.csv")
print("loans:", loans.shape)
print("bureau:", bureau.shape)

## Part 2: Assess — Visual

Before writing a single line of diagnostic code, just **look at the data**. Run each cell below and actually read the output — don't skim past it to get to the "real" work. A surprising amount of what's wrong with a dataset is visible just from `.head()` and a random `.sample()`, and visual inspection often catches things programmatic checks miss (like a plausible-but-wrong value, or a column name that doesn't match its content).

In [ ]:
loans.head(10)

In [ ]:
loans.sample(10, random_state=1)

In [ ]:
bureau.head()

**Pause here.** Before moving to the programmatic checks below, jot down anything that already looks off from just eyeballing the tables above — new markdown cell, your own words. (There's no solution cell for this one — the point is the noticing, not a specific answer. Compare your list to the full one at the end of this section.)

## Part 2: Assess — Programmatic

Now confirm and quantify what you spotted visually — and catch what you didn't.

**A1.** Run `loans.info()`. Which columns have fewer non-null values than the total row count? What are the columns' dtypes — does anything look like it should be a different type?

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
loans.info()

**A2.** Get the missing-value count per column into `missing_counts`, using `.isnull().sum()`.

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
missing_counts = loans.isnull().sum()
print(missing_counts)

**A3.** Check for duplicates two ways: `full_dupes` (exact duplicate rows, `.duplicated().sum()`) and `id_dupes` (duplicate `application_id` values specifically).

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
full_dupes = loans.duplicated().sum()
id_dupes = loans["application_id"].duplicated().sum()
print(full_dupes, id_dupes)

**A4.** Get every distinct value of `employment_status` into `employment_values`, using `.unique()`. Look closely — how many of these are actually the *same* category, just typed/cased differently?

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
employment_values = loans["employment_status"].unique()
print(sorted(employment_values.tolist()))

**A5.** Print `interest_rate`'s dtype and its first 5 values. Is this the dtype you'd want for a column you're going to do math on?

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
print(loans["interest_rate"].dtype, loans["interest_rate"].head(5).tolist())

**A6.** Get `numeric_summary`: `.describe()` on `applicant_age`, `annual_income`, and `loan_amount` together. Look at the `min` and `max` rows specifically — anything physically impossible in there?

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
numeric_summary = loans[["applicant_age", "annual_income", "loan_amount"]].describe()
print(numeric_summary)

**A7.** Print the first 3 values of `contact_info`. How many distinct pieces of information does each value actually contain?

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
print(loans["contact_info"].head(3).tolist())

**A8.** Compare the `application_id` values in `loans` versus `bureau` using set operations: `only_in_loans` (IDs in `loans` but not `bureau`) and `only_in_bureau` (the reverse). Print the count of each. What does this tell you about whether these two files describe the exact same population?

In [ ]:
# YOUR CODE HERE


**Solution**

In [ ]:
bureau_ids = set(bureau["application_id"])
loan_ids = set(loans["application_id"])
only_in_loans = loan_ids - bureau_ids
only_in_bureau = bureau_ids - loan_ids
print(len(only_in_loans), len(only_in_bureau))

### Full Assessment: Issue List

Here's the complete list your checks above should surface. Compare it to what you wrote down visually — did the programmatic checks catch anything you missed, or vice versa?

**Tidiness issues** (structural — how the data is organized):
1. `loans` and `bureau` describe the same observational unit (an applicant) but live in two separate tables — they should be one table.
2. `contact_info` bundles **two variables** (phone number and region) into a single column.

**Quality issues** (content — what's actually wrong with the values):
1. `annual_income` has 15 missing values; `credit_score` has ~10-11; `interest_rate` has ~15 (some genuinely missing, some stored as the literal text `"N/A"`).
2. `internal_notes` is entirely empty — 100% missing, contributes nothing.
3. 8 exact duplicate rows.
4. 13 duplicate `application_id`s (re-submitted applications).
5. `employment_status` has inconsistent casing — `"Employed"`, `"employed"`, `"EMPLOYED"` are all meant to be the same category.
6. `interest_rate` is stored as text with a trailing `%` (e.g. `"5.72%"`), not a usable number.
7. A handful of physically implausible values: negative `annual_income`, negative `loan_amount`, and `applicant_age` values of `999` and `14` (well outside a plausible adult borrower range).
8. `credit_score` is stored as `float64` only because missing values forced it out of integer form — it should be a whole number once the missing values are handled.

Two things you'll notice in the numbers *aren't* on this list, on purpose: after merging, some applicants have no bureau record at all (`bureau_score` and friends will be `NaN` for them), and most applicants have no delinquency date. Both of those `NaN`s are **meaningful information** ("no bureau hit" / "never delinquent"), not something to clean away — exactly the distinction Phase 6 covered.

## Part 3: Clean

Work through each issue with **Define → Code → Test**. `Test` here means real `assert` statements — if an assertion fails, your fix didn't fully work, and you find out immediately instead of discovering it three notebooks later.

Run this setup cell first — clean copies to work from, so `loans`/`bureau` above stay untouched as your "what it originally looked like" reference:

In [ ]:
loans_clean = loans.copy()
bureau_clean = bureau.copy()

### Tidiness Issue 1 — two tables, one observational unit

**Define:** merge `loans_clean` and `bureau_clean` on `application_id`, keeping every loan application (`how="left"`) even if it has no matching bureau record.

In [ ]:
# YOUR CODE HERE (Code)


**Solution**

In [ ]:
# Code
combined = pd.merge(loans_clean, bureau_clean, on="application_id", how="left")

# Test
assert len(combined) == len(loans_clean), "merge should not change row count with a left join"
assert "bureau_score" in combined.columns
print("passed:", combined.shape)

### Tidiness Issue 2 — `contact_info` bundles two variables

**Define:** split `contact_info` on `" | "` into two new columns, `phone_number` and `region`, then drop the original bundled column.

In [ ]:
# YOUR CODE HERE (Code)


**Solution**

In [ ]:
# Code
combined[["phone_number", "region"]] = combined["contact_info"].str.split(
    " | ", expand=True, regex=False
)
combined = combined.drop(columns=["contact_info"])

# Test
assert "contact_info" not in combined.columns
assert combined["region"].isin(["North", "South", "East", "West"]).all()
print("passed:", combined[["phone_number", "region"]].head(3).to_dict("records"))

### Quality Issue 1 — `internal_notes` is entirely empty

**Define:** drop the column outright — an all-missing column carries zero information.

In [ ]:
# YOUR CODE HERE (Code)


**Solution**

In [ ]:
# Code
combined = combined.drop(columns=["internal_notes"])

# Test
assert "internal_notes" not in combined.columns
print("passed")

### Quality Issue 2 — exact duplicate rows

**Define:** drop exact duplicates.

In [ ]:
# YOUR CODE HERE (Code)


**Solution**

In [ ]:
# Code
before = len(combined)
combined = combined.drop_duplicates()
after = len(combined)

# Test
assert combined.duplicated().sum() == 0
print("passed:", before, "->", after)

### Quality Issue 3 — duplicate `application_id`s

**Define:** for re-submitted applications, the most recent submission is the one that matters — sort by `application_date`, then keep the **last** occurrence of each `application_id`.

In [ ]:
# YOUR CODE HERE (Code)


**Solution**

In [ ]:
# Code
combined = combined.sort_values("application_date")
combined = combined.drop_duplicates(subset=["application_id"], keep="last")

# Test
assert combined["application_id"].duplicated().sum() == 0
print("passed:", combined.shape)

### Quality Issue 4 — inconsistent `employment_status` casing

**Define:** standardize every value to title case (`"Self-Employed"`, not `"self-employed"` or `"SELF-EMPLOYED"`).

In [ ]:
# YOUR CODE HERE (Code)


**Solution**

In [ ]:
# Code
combined["employment_status"] = combined["employment_status"].str.title()

# Test
assert set(combined["employment_status"].unique()) == {
    "Employed", "Self-Employed", "Unemployed", "Retired"
}
print("passed:", sorted(combined["employment_status"].unique().tolist()))

### Quality Issue 5 — `interest_rate` stored as text

**Define:** strip the `%` and convert to numeric, letting `errors="coerce"` turn anything unparseable (like the literal text `"N/A"`) into `NaN` instead of crashing.

In [ ]:
# YOUR CODE HERE (Code)


**Solution**

In [ ]:
# Code
combined["interest_rate"] = pd.to_numeric(
    combined["interest_rate"].astype(str).str.replace("%", "", regex=False), errors="coerce"
)

# Test
assert pd.api.types.is_numeric_dtype(combined["interest_rate"])
print("passed:", combined["interest_rate"].dtype)

### Quality Issue 6 — missing values in `annual_income`, `credit_score`, `interest_rate`, `bureau_score`

**Define:** impute each with its own column median — a defensible default when there's no stronger business rule (like the group-aware imputation from Phase 3) available here.

In [ ]:
# YOUR CODE HERE (Code)


**Solution**

In [ ]:
# Code
income_median = combined["annual_income"].median()
score_median = combined["credit_score"].median()
rate_median = combined["interest_rate"].median()
combined["annual_income"] = combined["annual_income"].fillna(income_median)
combined["credit_score"] = combined["credit_score"].fillna(score_median)
combined["interest_rate"] = combined["interest_rate"].fillna(rate_median)
combined["bureau_score"] = combined["bureau_score"].fillna(combined["bureau_score"].median())

# Test
assert combined[["annual_income", "credit_score", "interest_rate"]].isnull().sum().sum() == 0
print("passed")

### Quality Issue 7 — implausible values

**Define:** negative `annual_income`/`loan_amount` are impossible — clip at 0. An `applicant_age` outside a plausible adult range (18-100) is a data entry error, not a real value — treat it as missing, then impute with the median (the same pattern as Issue 6, applied after flagging the bad values).

In [ ]:
# YOUR CODE HERE (Code)


**Solution**

In [ ]:
# Code
combined["annual_income"] = combined["annual_income"].clip(lower=0)
combined["loan_amount"] = combined["loan_amount"].clip(lower=0)
invalid_age_mask = ~combined["applicant_age"].between(18, 100)
combined.loc[invalid_age_mask, "applicant_age"] = np.nan
combined["applicant_age"] = combined["applicant_age"].fillna(combined["applicant_age"].median())

# Test
assert (combined["annual_income"] >= 0).all()
assert (combined["loan_amount"] >= 0).all()
assert combined["applicant_age"].between(18, 100).all()
print("passed")

### Quality Issue 8 — `credit_score`/`applicant_age` should be integers

**Define:** now that missing values are gone, cast both back to a plain integer dtype (they were only floats because `NaN` forced that earlier).

In [ ]:
# YOUR CODE HERE (Code)


**Solution**

In [ ]:
# Code
combined["credit_score"] = combined["credit_score"].round().astype(int)
combined["applicant_age"] = combined["applicant_age"].round().astype(int)

# Test
assert str(combined["credit_score"].dtype).startswith("int")
assert str(combined["applicant_age"].dtype).startswith("int")
print("passed:", combined["credit_score"].dtype, combined["applicant_age"].dtype)

## Final Validation

One last full pass, checking everything you fixed and confirming what's *supposed* to still show `NaN`.

In [ ]:
combined.info()
print("Remaining nulls:")
print(combined.isnull().sum()[combined.isnull().sum() > 0])
print("Duplicate rows:", combined.duplicated().sum())
print("Duplicate application_ids:", combined["application_id"].duplicated().sum())
print("Final shape:", combined.shape)

Notice `num_bureau_inquiries`, `open_credit_lines`, and `last_delinquency_date` still have `NaN`s — and that's correct, not a leftover bug. Those rows are applicants with no bureau record at all, or applicants who've simply never been delinquent. Cleaning isn't "remove every `NaN` you can find" — it's fixing what's actually wrong while leaving genuinely meaningful missingness alone. Knowing which is which is the actual judgment call at the center of this whole workflow.

## ✅ Checkpoint

**What you covered:**
- **Gather**: loading from more than one source into a shared working set
- **Assess**: visual inspection first, then programmatic checks (`.info()`, `.isnull().sum()`, `.duplicated()`, `.unique()`, `.describe()`, set comparisons across tables) to confirm and quantify what you saw — and to catch what visual inspection alone would miss
- Distinguishing **tidiness issues** (structural — split tables, bundled columns) from **quality issues** (content — missing values, duplicates, bad dtypes, inconsistent categories, implausible values)
- **Clean**: Define → Code → Test for 2 tidiness issues and 8 quality issues, using techniques from Phases 3, 4, 6, and 7 — but this time applied as a deliberate, documented process instead of isolated exercises
- Recognizing that some `NaN`s are the correct final state, not a cleaning failure

**Why it matters for the project:** this Gather → Assess → Clean cycle, with a written issue list before any fix is coded, *is* the actual day-to-day job — much more so than any single pandas method on its own. Every technique from Phases 0-8 was really just building the vocabulary for this workflow.

**What's next:** matplotlib and seaborn, whenever you're ready — visualizing this same kind of cleaned data is the natural next step.